In [1]:
import preprocessamento as pp
import e2_engine_signals as es
import e2_backtest as bt

import pandas as pd
pd.set_option('display.max_columns', None)
import yfinance as yf



In [2]:
from datetime import datetime, timedelta

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
base = datetime.today().date() - timedelta(days=1)

In [5]:
def generate_performance_report(trades_df, initial_capital=100000):
    if trades_df.empty:
        return "Nenhum trade realizado no período."

    # Métricas Básicas
    wins = trades_df[trades_df['profit_pct'] > 0]
    losses = trades_df[trades_df['profit_pct'] <= 0]
    
    win_rate = len(wins) / len(trades_df)
    profit_factor = wins['profit_pct'].sum() / abs(losses['profit_pct'].sum()) if not losses.empty else np.inf
    avg_trade = trades_df['profit_pct'].mean()
    
    # Cálculo de Equity Curve
    trades_df['cum_profit'] = (1 + trades_df['profit_pct']).cumprod()
    final_return = (trades_df['cum_profit'].iloc[-1] - 1) * 100

    metrics = {
        "Taxa de Acerto (%)": round(win_rate * 100, 2),
        "Profit Factor": round(profit_factor, 2),
        "Retorno Total (%)": round(final_return, 2),
        "Média por Trade (%)": round(avg_trade * 100, 2),
        "Total de Operações": len(trades_df)
    }
    
    return pd.Series(metrics)


In [6]:
import matplotlib.pyplot as plt

def plot_equity_curve(df_original, trades_df):
    # Criar série diária de retornos da estratégia
    df_original['strategy_returns'] = 0.0
    for _, trade in trades_df.iterrows():
        df_original.loc[trade['exit_date'], 'strategy_returns'] = trade['profit_pct']
    
    # Retornos Acumulados
    equity_curve = (1 + df_original['strategy_returns']).cumprod()
    ibov_returns = df_original['adj_close_ibov'] / df_original['adj_close_ibov'].iloc[0]
    
    plt.figure(figsize=(12, 6))
    plt.plot(equity_curve, label='Estratégia (Scripts Python)', color='green')
    plt.plot(ibov_returns, label='IBOV (Benchmark)', color='gray', linestyle='--')
    plt.title('Evolução do Patrimônio: Estratégia vs. IBOV')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

import pandas as pd
from datetime import datetime, timedelta
import palpiteiro.scripts.e2_engine_signals_old as es

def run_automated_screener(tickers_list):
    """
    Percorre a lista de papéis, baixa os dados e testa todas as estratégias.
    """
    base_date = datetime.today().date() - timedelta(days=1)
    results = []
    
    # Mapeamento das ferramentas da sua caixa
    strategies = {
        # Grupo 1: Estratégias de Explosão (Usam Backtest de Tendência/Trailing Stop)
        'Tendência': (es.strategy_trend, bt.run_backtest_trend_following),
        'Breakout': (es.strategy_breakout, bt.run_backtest_trend_following),
        
        # Grupo 2: Estratégias de Tiro Curto (Precisam de Backtest com Alvo Fixo ou Saída na Média)
        # OBS: Você precisa garantir que run_backtest_with_target esteja calibrado para scalps
        'Retorno_Média': (es.strategy_mean_reversion, bt.run_backtest_with_target), 
        'IFR2': (es.strategy_ifr2_quant, bt.run_backtest_with_target),
        'Momentum': (es.strategy_momentum_swing, bt.run_backtest_with_target) 
    }

    print(f"--- Iniciando Screening Estratégico em {len(tickers_list)} ativos ---")

    for papel in tickers_list:
        try:
            # 1. Download On-the-fly usando sua função montar_dataset
            # O sufixo .SA é importante para o Yahoo Finance/yfinance
            ticker_formatted = f"{papel}.SA" if ".SA" not in papel else papel
            df_base = pp.montar_dataset(ticker_formatted, dias_total_historico=1600, data_base=base_date)
            df_base = pp.adicionar_features_sniper(df_base)
            
            if df_base is None or df_base.empty:
                continue

            for nome_estrat, (func_estrat, func_backtest) in strategies.items():
                # 2. Aplicar Estratégia
                df_with_signals = func_estrat(df_base.copy())
                
                # 3. Rodar Backtest (Assume-se que run_backtest e report estão no escopo)
                #trades = bt.run_backtest_with_target(df_with_signals)
                trades = func_backtest(df_with_signals)
                print(trades.head())
                
                if not trades.empty:
                    metrics = generate_performance_report(trades)
                    
                    results.append({
                        'Ativo': papel,
                        'Estratégia': nome_estrat,
                        'Expectativa_%': metrics['Média por Trade (%)'],
                        'Profit_Factor': metrics['Profit Factor'],
                        'Taxa_Acerto_%': metrics['Taxa de Acerto (%)'],
                        'Total_Trades': metrics['Total de Operações']
                    })
                    
            print(f"Sucesso: {papel} processado.")
            
        except Exception as e:
            raise e
        
            print(f"Erro ao processar {papel}: {str(e)}")

    # Consolidar Ranking
    ranking_df = pd.DataFrame(results)
    if not ranking_df.empty:
        # Ordenamos pelos melhores ganhos médios por trade
        print(f'--- Screening Completo: {len(ranking_df)} resultados obtidos ---')
        ranking_df = ranking_df.sort_values(by='Expectativa_%', ascending=False)
    else:
        print('--- Screening Completo: Nenhum resultado obtido ---')
        
    return ranking_df

In [48]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def run_automated_screener(tickers_list):
    base_date = datetime.today().date() - timedelta(days=1)
    results = []
    
    # 1. Configuração de Janelas de Estresse
    # Usaremos os últimos 1600 dias (~6 anos). 
    # Destes, os últimos 300 dias serão o nosso "Out-of-Sample" (Teste Cego).
    DIAS_OOS = 300 
    
    strategies = {
        'Tendência': (es.strategy_trend, bt.run_backtest_trend_following),
        'Breakout': (es.strategy_breakout, bt.run_backtest_trend_following),
        'Retorno_Média': (es.strategy_mean_reversion, bt.run_backtest_with_target), 
        'IFR2': (es.strategy_ifr2_quant, bt.run_backtest_with_target),
        'Momentum': (es.strategy_momentum_swing, bt.run_backtest_with_target) 
    }

    for papel in tickers_list:
        try:
            ticker_formatted = f"{papel}.SA" if ".SA" not in papel else papel
            df_full = pp.montar_dataset(ticker_formatted, dias_total_historico=1600, data_base=base_date)
            df_full = pp.adicionar_features_sniper(df_full)
            print(f"Processando {papel} - Total de dias: {len(df_full)}")
            if df_full is None or df_full.empty: continue

            # Divisão dos dados para validação
            split_idx = len(df_full) - DIAS_OOS
            df_insample = df_full.iloc[:split_idx] # Treino
            df_outsample = df_full.iloc[split_idx:] # Teste Real

            for nome_estrat, (func_estrat, func_backtest) in strategies.items():
                # --- TESTE 1: IN-SAMPLE (O passado que selecionamos) ---
                df_is_signals, checklist = func_estrat(df_insample.copy())
                
                trades_is = func_backtest(df_is_signals)
                
                if trades_is.empty or len(trades_is) < 15: # Mínimo de trades para ter relevância
                    continue
                
                metrics_is = generate_performance_report(trades_is)

                # --- TESTE 2: OUT-OF-SAMPLE (O "Pulo do Gato") ---
                # Testamos a MESMA estratégia no período recente sem mudar nenhum parâmetro
                df_oos_signals, checklist = func_estrat(df_outsample.copy())
                trades_oos = func_backtest(df_oos_signals)
                
                exp_is=0
                degradacao=0
                if trades_oos.empty:
                    performance_drop = 1.0 # 100% de queda se não operou
                else:
                    metrics_oos = generate_performance_report(trades_oos)
                    # Calculamos a degradação: se a expectativa caiu mais de 50%, é lixo (overfitting)
                    exp_is = metrics_is['Média por Trade (%)']
                    exp_oos = metrics_oos['Média por Trade (%)']
                    degradacao = (exp_oos / exp_is) if exp_is > 0 else 0

                # --- FILTRO DE ROBUSTEZ ---
                # Só aceitamos se:
                # 1. Tiver lucro no In-Sample
                # 2. A performance no Out-of-Sample for pelo menos 40% da original
                # 3. Profit Factor > 1.2
                
                passou_stress = (exp_is > 0.5) and (degradacao > 0.4) and (metrics_is['Profit Factor'] > 1.2)

                if passou_stress:
                    results.append({
                        'Ativo': papel,
                        'Estratégia': nome_estrat,
                        'Expectativa_IS_%': exp_is,
                        'Expectativa_OOS_%': exp_oos,
                        'Robustez_Score': round(degradacao * 100, 2), # % de manutenção da performance
                        'Profit_Factor': metrics_is['Profit Factor'],
                        'Total_Trades': len(trades_is) + len(trades_oos)
                    })
                else:
                    print(f"{papel} - {nome_estrat}: Rejeitado no teste de robustez. Degradação: {round(degradacao * 100, 2)}%")
            
            print(f"Sucesso: {papel} validado.")
            
        except Exception as e:
            print(f"Erro ao processar {papel}: {str(e)}")
            raise e

    ranking_df = pd.DataFrame(results)
    if not ranking_df.empty:
        # Ordenamos pela Robustez e depois pela Expectativa
        ranking_df = ranking_df.sort_values(by=['Robustez_Score', 'Expectativa_OOS_%'], ascending=False)
    
    return ranking_df

In [49]:
# Lista ampliada para aumentar a amostragem e validar o IFR2
papeis_para_teste = [
    # --- Blue Chips & Financeiro (Ideais para Retorno à Média/IFR2) ---
    'ITUB4', 'BBDC4', 'BBAS3', 'ABEV3', 'VIVT3', 'SANB11', 'B3SA3', 'SBSP3',
    
    # --- Commodities & Indústria (Tendência e Breakouts) ---
    'VALE3', 'PETR4', 'CSNA3', 'VBBR3', 'GGBR4', 'SUZB3', 'KLBN11', 
    
    # --- Utilidade Pública/Elétricas (Inércia e Dividendos) ---
    'EGIE3', 'CPFE3', 'ENGI11',
    
    # --- Consumo, Varejo & Agro (Alta Volatilidade) ---
    'LREN3', 'MGLU3', 'RAIZ4', 'BRFS3', 'BEEF3',  'HYPE3',
    
    # --- Tecnologia e Crescimento ---
    'WEGE3', 'EQTL3', 'RADL3', 'TOTS3'
]

In [50]:
# 1. Executar o Screener
ranking_geral = run_automated_screener(papeis_para_teste)

[ITUB4.SA] Cache existente cobre de 2020-01-02 a 2026-02-10.
[ITUB4.SA] Cache NÃO cobre período solicitado. Download total necessário.
[ITUB4.SA] Iniciando download de dados (lógica de atualização/complemento)...
[ITUB4.SA] Período de download yfinance: 2020-01-01 a 2026-02-14
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[ITUB4.SA] Realizando verificação de divergência entre cache antigo e dados frescos...
[ITUB4.SA] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
[ITUB4.SA] Cache principal ATUALIZADO com dados frescos: /palpiteiro/dados\ITUB4.csv
[ITUB4.SA] Retornando 1521 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20180716_20260210.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2026-02-10.
[^BVSP] Cache NÃO cobre período solicitado. Download total necessário.
[^BVSP] Iniciando download de dados (lógica de atua


1 Failed download:
['CSNA3.SA']: Timeout('Failed to perform, curl: (28) Connection timed out after 10002 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')


[CSNA3.SA] AVISO: Nenhum dado retornado pelo yfinance para o download.
[CSNA3.SA] Usando cache do disco (se existia) devido à falha no download.
[CSNA3.SA] Retornando 1521 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20180716_20260210.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2026-02-10.
[^BVSP] Cache NÃO cobre período solicitado. Download total necessário.
[^BVSP] Iniciando download de dados (lógica de atualização/complemento)...
[^BVSP] Período de download yfinance: 2020-01-01 a 2026-02-14
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Realizando verificação de divergência entre cache antigo e dados frescos...
[^BVSP] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
[^BVSP] Cache principal ATUALIZADO com dados frescos: /palpiteiro/dados\^BVSP.csv
[^BVSP] Retornando 1521 linhas para o período solicitado


1 Failed download:
['BRFS3.SA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-01-01 -> 2026-02-15) (Yahoo error = "No data found, symbol may be delisted")')


[BRFS3.SA] AVISO: Nenhum dado retornado pelo yfinance para o download.
[BRFS3.SA] Usando cache do disco (se existia) devido à falha no download.
[BRFS3.SA] Retornando 1405 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20180716_20260210.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2026-02-10.
[^BVSP] Cache NÃO cobre período solicitado. Download total necessário.
[^BVSP] Iniciando download de dados (lógica de atualização/complemento)...
[^BVSP] Período de download yfinance: 2020-01-01 a 2026-02-14
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Realizando verificação de divergência entre cache antigo e dados frescos...
[^BVSP] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
[^BVSP] Cache principal ATUALIZADO com dados frescos: /palpiteiro/dados\^BVSP.csv
[^BVSP] Retornando 1521 linhas para o período solicitado

In [51]:
ranking_geral.head()

""


In [13]:
# 2. Filtrar apenas o "Filé Mignon"
# Estratégias com Expectativa positiva e mais de 5 trades para ter relevância
oportunidades_reais = ranking_geral[
    (ranking_geral['Expectativa_%'] > 0) & 
    (ranking_geral['Total_Trades'] >= 5)
]

print("\n--- TOP 10 MELHORES COMBINAÇÕES ATIVO x ESTRATÉGIA ---")
print(oportunidades_reais.head(10))

KeyError: 'Expectativa_%'

In [ ]:
import pandas as pd
import numpy as np
import e2_risk_manager as rm
import preprocessamento as pp

# --- ⚙️ PARÂMETROS DO GESTOR (O COFRE) ---
CAPITAL_TOTAL = 100000       # Seu patrimônio total disponível para trade
RISCO_POR_TRADE = 0.01       # 1% de risco per capita (R$ 1.000,00)
ATR_MULTIPLIER_STOP = 2.0    # Referência para volatilidade do stop

# --- 🛡️ FILTROS DE QUALIDADE (O CRIVO) ---
MIN_TRADES_VALIDOS = 6       # Menos que isso é sorte, não estatística.
MIN_EXPECTATIVA_PCT = 0.50   # Se a expectativa média por trade for < 0.5%, nem levanta da cama.

base_date = datetime.today().date() - timedelta(days=1)

print(f"--- 🛡️ GESTÃO DE PORTFÓLIO E SIZING AUTOMÁTICO 🛡️ ---")
print(f"Capital: R$ {CAPITAL_TOTAL:,.2f} | Risco Aceitável/Trade: R$ {CAPITAL_TOTAL * RISCO_POR_TRADE:,.2f}")
print(f"Filtros: Mínimo {MIN_TRADES_VALIDOS} trades históricos e Expectativa > {MIN_EXPECTATIVA_PCT}%")

# 1. Aplicando o Filtro de Viabilidade no DataFrame Geral
# Usamos o ranking_geral para garantir que estamos olhando para tudo, e filtramos aqui.
candidatos = ranking_geral[
    (ranking_geral['Total_Trades'] >= MIN_TRADES_VALIDOS) &
    (ranking_geral['Expectativa_%'] >= MIN_EXPECTATIVA_PCT)
].copy()

print(f"\n🔎 {len(ranking_geral)} estratégias analisadas -> {len(candidatos)} aprovadas pelos filtros.")

alocacoes = []

for index, row in candidatos.iterrows():
    ativo = row['Ativo']
    estrategia = row['Estratégia']
    expectativa = row['Expectativa_%']
    
    try:
        # 2. Obter Dados Frescos (Preço e Volatilidade de Hoje)
        ticker_formatted = f"{ativo}.SA" if ".SA" not in ativo else ativo
        
        # Baixa histórico curto (100 dias) apenas para calcular o ATR recente e pegar preço atual
        df_recent = pp.montar_dataset(ticker_formatted, dias_total_historico=100, data_base=base_date)
        
        if df_recent is None or df_recent.empty:
            print(f"⚠️ Dados indisponíveis para {ativo}. Pulando.")
            continue
            
        # Garante cálculo do ATR
        if 'atr' not in df_recent.columns:
            df_recent = pp.adicionar_features_sniper(df_recent)
            
        ultimo_fechamento = df_recent['adj_close'].iloc[-1]
        ultimo_atr = df_recent['atr'].iloc[-1]
        data_referencia = df_recent.index[-1].strftime('%Y-%m-%d')
        
        # 3. Simulação do Stop Loss pela Volatilidade (Padronização do Risco)
        stop_estimado_price = ultimo_fechamento - (ultimo_atr * ATR_MULTIPLIER_STOP)
        
        # 4. Cálculo do Sizing
        sizing = rm.calculate_position_size(
            account_balance=CAPITAL_TOTAL, 
            risk_pct=RISCO_POR_TRADE, 
            entry_price=ultimo_fechamento, 
            stop_loss_price=stop_estimado_price
        )
        
        # 5. Compilação
        alocacoes.append({
            'Ativo': ativo,
            'Estratégia': estrategia,
            'Data_Ref': data_referencia,
            'Preço_Atual': round(ultimo_fechamento, 2),
            'Stop_Ref': round(stop_estimado_price, 2),
            'Qtd_Ações': sizing['quantity'],
            'Risco_Financeiro_R$': sizing['risk_capital'],
            'Investimento_Total_R$': sizing['financial_exposure'],
            '%_Carteira': round((sizing['financial_exposure'] / CAPITAL_TOTAL) * 100, 1),
            'Exp_Retorno_%': expectativa,
            'Histórico_Trades': row['Total_Trades']
        })
        
    except Exception as e:
        print(f"❌ Erro ao calcular {ativo}: {str(e)}")

# --- RELATÓRIO FINAL DE EXECUÇÃO ---
if alocacoes:
    df_alocacao = pd.DataFrame(alocacoes)
    
    # Ordena pelas melhores expectativas
    df_alocacao = df_alocacao.sort_values(by='Exp_Retorno_%', ascending=False)
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    print("\n=== 🚀 ORDENS SUGERIDAS PARA AMANHÃ ===")
    display(df_alocacao[[
        'Ativo', 'Estratégia', 'Qtd_Ações', 'Preço_Atual', 'Stop_Ref', 
        'Investimento_Total_R$', '%_Carteira', 'Exp_Retorno_%'
    ]])
    
    print(f"\nNOTA: O 'Stop_Ref' é calculado com {ATR_MULTIPLIER_STOP}x ATR atual para definir o tamanho da mão.")
    print("Se a estratégia der sinal de entrada amanhã, use essa quantidade de ações.")
else:
    print("\n⚠️ Nenhum ativo passou nos critérios rigorosos de filtro.")

--- 🛡️ GESTÃO DE PORTFÓLIO E SIZING AUTOMÁTICO 🛡️ ---
Capital: R$ 100,000.00 | Risco Aceitável/Trade: R$ 1,000.00
Filtros: Mínimo 6 trades históricos e Expectativa > 0.5%


KeyError: 'Total_Trades'

In [ ]:
# Célula final do notebook de screening
if 'df_alocacao' in locals() and not df_alocacao.empty:
    # Salvamos apenas o necessário para o robô diário
    cols_to_save = ['Ativo', 'Estratégia', 'Exp_Retorno_%', 'Histórico_Trades']
    df_alocacao[cols_to_save].to_csv('dados/meu_portfolio.csv', index=False)
    print("✅ Portfólio salvo com sucesso em 'dados/meu_portfolio.csv'. O robô já pode ler isso.")
else:
    print("⚠️ Nenhum portfólio gerado para salvar.")

✅ Portfólio salvo com sucesso em 'dados/meu_portfolio.csv'. O robô já pode ler isso.


: 

: 

: 

: 

: 

: 